# GradMem Idea 3 on Colab

This notebook clones your fork, installs packages, mounts Google Drive and runs one training configuration.

How it survives the end of a Colab session
- Code comes from GitHub, packages are reinstalled, and MQAR data is regenerated from a fixed seed.
- The run trains on the fast Colab disk. Every 10 minutes the newest complete checkpoints, the TensorBoard logs and the console log are mirrored to Drive.
- In a new session, run all cells again with the same `RUN`. The helper copies the checkpoints back from Drive and the run continues from the last saved step.

The repo script calls `trainer.train()` without resume. The helper `colab_run.py` adds resume from the outside, so the repo code stays unchanged.

Use a GPU runtime (Runtime, Change runtime type, T4 GPU).

## 1. Settings

Change `FORK_URL` to your fork. For a private fork, add a GitHub token as a Colab Secret named `GITHUB_TOKEN` (key icon on the left). Never paste a token into a cell.

In [ ]:
FORK_URL = 'https://github.com/alex-semin-ai/gradmem.git'  # your fork
BRANCH = 'my-idea3'                                        # your branch in the fork
PROJ = '/content/drive/MyDrive/gradmem_idea3'              # project folder on Drive
REPO = '/content/gradmem'                                  # code on the Colab disk
LOCAL_RUNS = '/content/runs'                               # fast local training folders
SYNC_MINUTES = 10                                          # how often runs are mirrored to Drive

## 2. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
for d in ['runs', 'results']:
    os.makedirs(f'{PROJ}/{d}', exist_ok=True)
os.makedirs(LOCAL_RUNS, exist_ok=True)
print('Drive project folder:', PROJ)

## 3. Get the code

Clones the fork on the first run of the session and pulls the newest commits on later runs. The commit hash is printed so every run records which code produced it.

In [ ]:
import os, subprocess

def clone_url():
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    if token:
        return FORK_URL.replace('https://', f'https://x-access-token:{token}@')
    return FORK_URL

if not os.path.isdir(f'{REPO}/.git'):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, clone_url(), REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q', 'origin', BRANCH], check=True)

COMMIT = subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'],
                        capture_output=True, text=True, check=True).stdout.strip()
print('Code version:', BRANCH, COMMIT)

## 4. Install packages and check the GPU

The repo has no `requirements.txt`. Its conda environment uses transformers 4.51.3, so that version is pinned here. Colab's own torch is kept, which avoids a large download. Other preinstalled Colab packages may print version warnings, and those can be ignored.

No Hugging Face token is needed. The model config is built locally by `llama_config.py` in the repo, so nothing is downloaded from the gated `meta-llama/Llama-3.2-1B` repository.

The two environment variables tell transformers to skip the preinstalled TensorFlow. Training uses PyTorch only. The training process started in cell 7 inherits these variables.

In [ ]:
!pip install -q transformers==4.51.3

import os
os.environ['USE_TF'] = '0'     # transformers skips the preinstalled TensorFlow (training uses PyTorch)
os.environ['USE_TORCH'] = '1'

import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
assert torch.cuda.is_available(), 'No GPU. Switch the runtime to T4 GPU.'
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 5. Write the resume helper

`colab_run.py` does four things.
1. Copies the run's checkpoints and logs from Drive to the local disk, if the local disk has none.
2. Makes `Trainer.train()` resume from the newest complete local checkpoint.
3. Mirrors complete checkpoints, TensorBoard logs and the console log to Drive every `SYNC_MINUTES`.
4. Mirrors once more when training ends, fails or is stopped.
5. Refuses to start a run that breaks the setup rules. A `gradmem` run must start from an RMT checkpoint of the same size trained with the reconstruction loss, which the checkpoint's own `config.json` shows (`model_type` `rmt_2segm`). A `gradmem_reference` run must start from scratch, and an `rmt` run must use the reconstruction loss.
6. Stops the run at once if the starting checkpoint (`init_checkpoint`) leaves weights unloaded. The repo scripts would otherwise only log this and quietly train from scratch.
7. Keeps a `summary.json` in the run folder and rebuilds `results/results.csv` for all runs at every mirror.
8. Writes a launch entry and an end entry for the run in `results/runs.md`.

If Colab disconnects, that session gets no end entry, and the run shows `running` with an old `last_update`. The next session of the same run adds a new launch entry and continues the log.

A checkpoint counts as complete only when its weights, optimizer, scheduler, RNG state and trainer state files all exist. A save cut by a disconnect is therefore never used for resume. The Trainer writes `trainer_state.json` last, so during training the mirror copies a checkpoint once that file is readable and the folder has been quiet for 3 seconds. Checkpoints are copied before the log files. Each copy to Drive goes to a temporary folder first and is renamed when done, and Drive always keeps the newest checkpoint it has received.

In [ ]:
%%writefile /content/colab_run.py
"""Run a repo training script with resume and Drive mirroring.

Usage
    python colab_run.py --repo R --script S --local_dir L --drive_dir D --log F --sync_minutes M -- <script args>
The script args must contain --exp_path equal to --local_dir.
"""
import argparse, csv, json, os, re, runpy, shutil, signal, sys, threading, time
from pathlib import Path

REQUIRED = ('trainer_state.json', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth')
MODEL_FILES = ('model.safetensors', 'pytorch_model.bin')


def step_of(p):
    m = re.fullmatch(r'checkpoint-(\d+)', p.name)
    return int(m.group(1)) if m else -1


def is_complete(p):
    ok = all((p / f).is_file() and (p / f).stat().st_size > 0 for f in REQUIRED)
    return ok and any((p / f).is_file() for f in MODEL_FILES)


def is_stable(p, seconds=3):
    """True when the Trainer has finished writing the checkpoint.

    The Trainer writes trainer_state.json last (after weights, optimizer, scheduler and RNG state),
    so a readable trainer_state.json plus a few quiet seconds means the folder is complete."""
    try:
        json.load(open(p / 'trainer_state.json'))
    except (OSError, ValueError):
        return False
    newest = max((f.stat().st_mtime for f in p.rglob('*') if f.is_file()), default=0)
    return time.time() - newest > seconds


def complete_ckpts(d, stable=False):
    if not d.is_dir():
        return []
    found = [p for p in d.iterdir() if p.is_dir() and step_of(p) >= 0 and is_complete(p)]
    if stable:
        found = [p for p in found if is_stable(p)]
    return sorted(found, key=step_of)


def copy_dir_atomic(src, dst):
    tmp = dst.with_name(dst.name + '.tmp')
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(src, tmp)
    shutil.rmtree(dst, ignore_errors=True)
    os.rename(tmp, dst)


def copy_if_newer(src, dst):
    if dst.exists():
        s, d = src.stat(), dst.stat()
        if s.st_size == d.st_size and s.st_mtime <= d.st_mtime:
            return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)


def restore(local, drive):
    """Copy checkpoints and files from Drive when the local run folder has no checkpoint."""
    if complete_ckpts(local) or not drive.is_dir():
        return
    for c in complete_ckpts(drive / 'checkpoints'):
        copy_dir_atomic(c, local / c.name)
    files = drive / 'files'
    if files.is_dir():
        for p in files.rglob('*'):
            if p.is_file():
                copy_if_newer(p, local / p.relative_to(files))
    n = len(complete_ckpts(local))
    if n:
        print(f'[colab_run] restored {n} checkpoint(s) from Drive', flush=True)


def mirror(local, drive, log, final=False):
    """Mirror complete checkpoints first, then all other files of the run, to Drive."""
    (drive / 'checkpoints').mkdir(parents=True, exist_ok=True)
    # 1. checkpoints. During training only checkpoints the Trainer finished writing are copied.
    local_ck = complete_ckpts(local, stable=not final)
    names = {c.name for c in local_ck}
    for c in local_ck:
        if not (drive / 'checkpoints' / c.name).is_dir():
            copy_dir_atomic(c, drive / 'checkpoints' / c.name)
    # remove a Drive checkpoint only when it is gone locally AND a newer one is already on Drive,
    # so Drive always keeps the newest checkpoint it has seen
    if local_ck:
        newest = max(step_of(c) for c in local_ck)
        for c in (drive / 'checkpoints').iterdir():
            if c.is_dir() and 0 <= step_of(c) < newest and c.name not in names:
                shutil.rmtree(c, ignore_errors=True)
    # 2. small files (TensorBoard logs, config, console log), one by one so one failure blocks nothing
    files = [(p, drive / 'files' / p.relative_to(local)) for p in local.rglob('*')
             if p.is_file() and not p.relative_to(local).parts[0].startswith('checkpoint-')]
    if log and Path(log).is_file():
        files.append((Path(log), drive / 'logs' / Path(log).name))
    for src, dst in files:
        try:
            copy_if_newer(src, dst)
        except OSError as e:
            print(f'[colab_run] could not copy {src.name}: {e}', flush=True)


# ---------- run log, summary.json, runs.md and results.csv ----------

def now():
    return time.strftime('%Y-%m-%d %H:%M')


def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2), encoding='utf8')
    os.replace(tmp, path)


def append_md(path, text):
    if not path:
        return
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, 'a', encoding='utf8') as f:
        f.write(text.rstrip('\n') + '\n\n')


NOT_METRICS = ('loss', 'runtime', 'per_second', 'epoch')


def eval_summary(log_history, best_metric, metric_name=None):
    """Accuracy key, best accuracy and the first eval step that reached 99%."""
    evals = [h for h in log_history if any(k.startswith('eval_') for k in h)]
    keys = sorted({k for h in evals for k in h
                   if k.startswith('eval_') and not any(x in k for x in NOT_METRICS)})
    key = None
    if metric_name:   # the metric the Trainer uses to pick the best model
        name = metric_name if metric_name.startswith('eval_') else f'eval_{metric_name}'
        key = name if name in keys else None
    for k in keys if key is None else []:   # else the key whose maximum equals the best metric
        vals = [h[k] for h in evals if k in h]
        if best_metric is not None and vals and abs(max(vals) - best_metric) < 1e-9:
            key = k
            break
    if key is None:
        named = [k for k in keys if any(x in k for x in ('acc', 'match', 'exact'))]
        key = named[0] if named else None
    if key is None:
        return None, None, None
    points = [(h.get('step', 0), h[key]) for h in evals if key in h]
    best = max(v for _, v in points)
    threshold = 0.99 if best <= 1.0 else 99.0
    hit = next((s for s, v in points if v >= threshold), None)
    return key, best, hit


def load_config(drive):
    p = drive / 'config.json'
    return json.load(open(p))['config'] if p.exists() else {}


def describe_settings(cfg):
    parts = [f"{cfg.get('num_kv_pairs')} pairs", f"K = {cfg.get('K')}", f"inner_lr {cfg.get('inner_lr')}",
             f"{cfg.get('n_layer')} layers", f"hidden {cfg.get('n_embd')}", f"seed {cfg.get('seed')}"]
    return ', '.join(p for p in parts if 'None' not in p)


def init_from(cfg):
    init = cfg.get('init_checkpoint') or cfg.get('init_base_checkpoint')
    if not init:
        return 'scratch'
    parts = Path(init).parts   # .../runs/<RUN>/<checkpoints/checkpoint-N | base>/model.safetensors
    if 'runs' in parts and parts.index('runs') + 2 < len(parts):
        i = parts.index('runs')
        return '/'.join(p for p in parts[i + 1:-1] if p != 'checkpoints')
    return init


class RunLog:
    """Keeps summary.json of one run up to date and writes its entries in runs.md."""

    def __init__(self, drive, note, commit, runs_md):
        self.drive, self.runs_md = drive, runs_md
        self.path = drive / 'summary.json'
        self.summary = json.load(open(self.path)) if self.path.exists() else {'run': drive.name, 'sessions': []}
        self.session = {'started': now(), 'commit': commit, 'note': note,
                        'start_step': None, 'end_step': None, 'seconds': 0.0}
        self.summary['sessions'].append(self.session)
        self.status, self.trainer, self.t0 = 'running', None, None

    def attach(self, trainer, start_step):
        self.trainer, self.t0 = trainer, time.time()
        self.session['start_step'] = start_step

    def update(self):
        s = self.summary
        s['status'], s['last_update'] = self.status, now()
        if self.trainer is not None:
            st = self.trainer.state
            self.session['end_step'] = st.global_step
            self.session['seconds'] = round(time.time() - self.t0, 1)
            key, best, hit = eval_summary(list(st.log_history), st.best_metric,
                                          getattr(self.trainer.args, 'metric_for_best_model', None))
            s.update(steps=st.global_step, metric=key, best=best, step_to_99=hit)
        self.drive.mkdir(parents=True, exist_ok=True)
        write_json_atomic(self.path, s)

    def launch_entry(self, resumed_from):
        cfg = load_config(self.drive)
        start = f'Resumed from {resumed_from}' if resumed_from else 'New run from step 0'
        append_md(self.runs_md, '\n'.join([
            f"### {now()} · {self.drive.name} · launch",
            f"- {start}. Initial weights from {init_from(cfg)}.",
            f"- Settings. {describe_settings(cfg)}.",
            f"- Code {self.session['commit']}.",
            f"- Why. {self.session['note'] or 'no note'}"]))

    def end_entry(self):
        s = self.summary
        best = s.get('best')
        best_txt = f"best accuracy {best:.4f}" if best is not None else 'no eval yet'
        hit = s.get('step_to_99')
        hit_txt = f'99% reached at step {hit}' if hit is not None else '99% not reached'
        done = self.session['end_step'] is not None and self.session['start_step'] is not None
        n = (self.session['end_step'] - self.session['start_step']) if done else 0
        speed = f"{1000 * self.session['seconds'] / n:.0f} ms per step in this session" if n > 0 else 'no steps in this session'
        append_md(self.runs_md, '\n'.join([
            f"### {now()} · {self.drive.name} · {self.status}",
            f"- Step {s.get('steps', 0)}, {best_txt}, {hit_txt}.",
            f"- {speed}.",
            "- Conclusion. Add it with the conclusion cell."]))


RESULT_COLUMNS = ['run', 'status', 'pairs', 'K', 'inner_lr', 'layers', 'hidden', 'seed', 'init_from',
                  'steps', 'best_acc', 'step_to_99', 'ms_per_step', 'sessions', 'code', 'last_update',
                  'why', 'conclusion']


def rebuild_results(runs_dir, csv_path):
    """Write results.csv from every run folder. Safe to call from several runs at once."""
    runs_dir, csv_path = Path(runs_dir), Path(csv_path)
    rows = []
    for d in sorted(p for p in runs_dir.iterdir() if p.is_dir()):
        cfg = load_config(d)
        s = json.load(open(d / 'summary.json')) if (d / 'summary.json').exists() else {}
        ses = s.get('sessions', [])
        done = [x for x in ses if x.get('start_step') is not None and x.get('end_step') is not None
                and x['end_step'] > x['start_step']]
        steps_done = sum(x['end_step'] - x['start_step'] for x in done)
        ms = round(1000 * sum(x['seconds'] for x in done) / steps_done, 1) if steps_done else None
        notes = [x['note'] for x in ses if x.get('note')]
        rows.append({
            'run': d.name, 'status': s.get('status', 'not started'),
            'pairs': cfg.get('num_kv_pairs'), 'K': cfg.get('K'), 'inner_lr': cfg.get('inner_lr'),
            'layers': cfg.get('n_layer'), 'hidden': cfg.get('n_embd'), 'seed': cfg.get('seed'),
            'init_from': init_from(cfg), 'steps': s.get('steps'), 'best_acc': s.get('best'),
            'step_to_99': s.get('step_to_99'), 'ms_per_step': ms, 'sessions': len(ses),
            'code': ses[-1].get('commit') if ses else None, 'last_update': s.get('last_update'),
            'why': notes[-1] if notes else None, 'conclusion': s.get('conclusion')})
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    tmp = csv_path.with_name(csv_path.name + f'.{os.getpid()}.tmp')
    with open(tmp, 'w', newline='', encoding='utf8') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_COLUMNS)
        w.writeheader()
        w.writerows(rows)
    os.replace(tmp, csv_path)
    return rows


GRADMEM_SCRIPT, RMT_SCRIPT = 'run_gradmemgpt_on_mqar.py', 'run_rmt_on_mqar.py'


def check_run_type(run_type, script, script_args):
    """Sanity check before anything starts. Returns an error message, or None when the run is allowed.

    gradmem            GradMem must start from an RMT checkpoint (plan, D1 and D10).
    gradmem_reference  GradMem from scratch, only for the reference runs E2 and E3.
    rmt                RMT always with the reconstruction loss (plan, D11). A warm start must come from RMT.
    The checkpoint's own config.json names the class that saved it, so it shows where the weights come from."""
    if not run_type:
        return None
    a = {script_args[i][2:]: script_args[i + 1] for i in range(len(script_args) - 1)
         if script_args[i].startswith('--')}
    init = a.get('init_checkpoint')

    def rmt_parent_problem():
        cfg_path = Path(init).parent / 'config.json'
        if not Path(init).is_file():
            return f'the starting checkpoint does not exist ({init})'
        if not cfg_path.is_file():
            return f'no config.json next to the starting checkpoint ({cfg_path})'
        c = json.load(open(cfg_path))
        if c.get('model_type') != 'rmt_2segm':
            return f'the starting checkpoint is not an RMT checkpoint (model_type {c.get("model_type")!r})'
        if not c.get('use_reconstruction_loss'):
            return 'the RMT parent was trained without the reconstruction loss (plan, D11)'
        base = c.get('base_config') or {}
        pairs = [('num_hidden_layers', 'n_layer'), ('hidden_size', 'n_embd'), ('vocab_size', 'vocab_size')]
        diff = [f'{arg} {base.get(key)} vs {a.get(arg)}' for key, arg in pairs
                if a.get(arg) is not None and str(base.get(key)) != str(a.get(arg))]
        if a.get('n_mem_tokens') is not None and str(c.get('n_mem_tokens')) != str(a.get('n_mem_tokens')):
            diff.append(f'n_mem_tokens {c.get("n_mem_tokens")} vs {a.get("n_mem_tokens")}')
        if diff:
            return 'the RMT parent has a different model size: ' + ', '.join(diff)
        return None

    if 'init_base_checkpoint' in a:
        return 'init_base_checkpoint is not allowed. It loads nothing from RMT checkpoints'
    if run_type == 'gradmem':
        if Path(script).name != GRADMEM_SCRIPT:
            return f'a gradmem run must use {GRADMEM_SCRIPT}, not {script}'
        if not init:
            return 'a gradmem run must start from RMT weights, but no init_checkpoint is given (plan, D1, D10)'
        return rmt_parent_problem()
    if run_type == 'gradmem_reference':
        if Path(script).name != GRADMEM_SCRIPT:
            return f'a gradmem_reference run must use {GRADMEM_SCRIPT}, not {script}'
        if init:
            return 'a gradmem_reference run trains from scratch, but init_checkpoint is given'
        return None
    if run_type == 'rmt':
        if Path(script).name != RMT_SCRIPT:
            return f'an rmt run must use {RMT_SCRIPT}, not {script}'
        if a.get('use_reconstruction_loss') != 'true':
            return 'an rmt run must use the reconstruction loss (plan, D11)'
        return rmt_parent_problem() if init else None
    return f'unknown run type {run_type!r}'


def main():
    argv = sys.argv[1:]
    split = argv.index('--')
    ap = argparse.ArgumentParser()
    ap.add_argument('--repo', required=True)
    ap.add_argument('--script', required=True)
    ap.add_argument('--local_dir', required=True)
    ap.add_argument('--drive_dir', required=True)
    ap.add_argument('--log', default='')
    ap.add_argument('--sync_minutes', type=float, default=10)
    ap.add_argument('--note', default='')
    ap.add_argument('--commit', default='')
    ap.add_argument('--runs_md', default='')
    ap.add_argument('--results_csv', default='')
    ap.add_argument('--run_type', default='', help='rmt, gradmem or gradmem_reference. Empty skips the check')
    a = ap.parse_args(argv[:split])
    script_args = argv[split + 1:]

    problem = check_run_type(a.run_type, a.script, script_args)
    if problem:
        print(f'[colab_run] REFUSED, {problem}. Nothing was started.', flush=True)
        sys.exit(2)
    if a.run_type:
        print(f'[colab_run] run type {a.run_type} checked', flush=True)

    local, drive = Path(a.local_dir), Path(a.drive_dir)
    local.mkdir(parents=True, exist_ok=True)
    restore(local, drive)

    runlog = RunLog(drive, a.note, a.commit, a.runs_md)
    existing = complete_ckpts(local)
    runlog.launch_entry(existing[-1].name if existing else None)
    runlog.update()

    import transformers
    original_train = transformers.Trainer.train
    training_started = {'value': False}

    # Fail fast when a starting checkpoint leaves weights unloaded. The repo scripts load
    # init_checkpoint with strict=False and only log missing keys, so a wrong checkpoint or flag
    # would otherwise train from scratch without any error. Only loads before training are
    # checked; the Trainer's own resume loads are left alone.
    try:
        import torch
    except ImportError:
        torch = None
    if torch is not None:
        original_load_state_dict = torch.nn.Module.load_state_dict

        def checked_load_state_dict(self, state_dict, strict=True, assign=False):
            result = original_load_state_dict(self, state_dict, strict=strict, assign=assign)
            if not strict and not training_started['value']:
                missing = [k for k in result.missing_keys if not k.endswith('lm_head.weight')]  # tied to embeddings
                if missing:
                    raise RuntimeError(
                        f'[colab_run] the starting checkpoint left {len(missing)} weights unloaded, '
                        f'for example {missing[:5]}. Check init_checkpoint, and never use '
                        f'init_base_checkpoint for an RMT parent.')
                print(f'[colab_run] starting checkpoint loaded, no missing weights '
                      f'({len(result.unexpected_keys)} unused keys)', flush=True)
            return result

        torch.nn.Module.load_state_dict = checked_load_state_dict

    def train_with_resume(self, *args, **kwargs):
        training_started['value'] = True
        start_step = 0
        if kwargs.get('resume_from_checkpoint') is None:
            ck = complete_ckpts(Path(self.args.output_dir))
            if ck:
                kwargs['resume_from_checkpoint'] = str(ck[-1])
                start_step = step_of(ck[-1])
                print(f'[colab_run] resuming from {ck[-1].name}', flush=True)
            else:
                print('[colab_run] no checkpoint found, starting from step 0', flush=True)
        runlog.attach(self, start_step)
        return original_train(self, *args, **kwargs)

    transformers.Trainer.train = train_with_resume

    # The repo scripts create exp_path with mkdir(parents=True) and stop if it exists.
    # The run folder always exists here (restored checkpoints, resume), so allow it for this one folder.
    import pathlib
    original_mkdir = pathlib.Path.mkdir

    def mkdir_allow_run_dir(self, mode=0o777, parents=False, exist_ok=False):
        if self.resolve() == local.resolve():
            exist_ok = True
        return original_mkdir(self, mode=mode, parents=parents, exist_ok=exist_ok)

    pathlib.Path.mkdir = mkdir_allow_run_dir

    lock, stop = threading.Lock(), threading.Event()

    def safe_mirror(final=False):
        with lock:
            try:
                runlog.update()
                if a.results_csv:
                    rebuild_results(drive.parent, a.results_csv)
                mirror(local, drive, a.log, final=final)
            except Exception as e:  # a failed mirror must not stop training
                print(f'[colab_run] mirror failed: {e}', flush=True)

    def loop():
        while not stop.wait(a.sync_minutes * 60):
            safe_mirror()

    threading.Thread(target=loop, daemon=True).start()
    signal.signal(signal.SIGTERM, lambda *_: sys.exit(143))   # so a stop still runs the final mirror

    os.chdir(a.repo)
    sys.path.insert(0, a.repo)
    sys.argv = [a.script] + script_args
    try:
        runpy.run_path(a.script, run_name='__main__')
        runlog.status = 'finished'
    except SystemExit as e:
        runlog.status = 'stopped' if e.code == 143 else ('finished' if e.code in (0, None) else f'failed (exit {e.code})')
        raise
    except BaseException as e:
        runlog.status = f'failed ({type(e).__name__})'
        raise
    finally:
        stop.set()
        safe_mirror(final=True)
        try:
            runlog.end_entry()
        except Exception as e:
            print(f'[colab_run] runs.md entry failed: {e}', flush=True)
        print(f'[colab_run] {runlog.status}, final mirror to Drive done', flush=True)


if __name__ == '__main__':
    main()

## 6. Choose a run

You choose the **type** of run, and the next cell builds the full configuration from one shared base. The base follows the plan's decisions, 4 layers, hidden 128, 16 pairs, 1 000 000 training examples (D12, D13).

| `RUN_TYPE` | What it is | Starts from |
|---|---|---|
| `'rmt'` | RMT, always with the reconstruction loss (D11). E1, E4, E5, and RMT pilots | scratch, or an RMT parent for warm starts (E4, E5 from E1) |
| `'gradmem'` | GradMem started from RMT weights, as the supervisor's scheme requires (D1, D10). E2r, E3r, E8r, and GradMem pilots | **an RMT parent, required** |
| `'gradmem_reference'` | GradMem from scratch. Only for the reference runs E2 and E3 | scratch |

`K` means passes over the context for RMT, and write steps for GradMem.

For a `'gradmem'` run, set `PARENT_RMT_RUN` to the name of a finished RMT run on Drive. The next cell checks that the parent really is an RMT run with the reconstruction loss and the same model size, takes its newest checkpoint (or `PARENT_STEP`), and sets `init_checkpoint`. A `'gradmem'` run without a valid parent does not start.

`EXTRA` adds or changes script arguments for this run only, for example `max_steps` for a pilot or `learning_rate` for a sweep. The default pilot also tests a linearly decaying learning rate (`lr_scheduler_type='linear'`). The repo's Trainer lets it fall to a tenth of the rate by the last step, instead of keeping it constant. If the pilot ends higher and steadier than the constant-rate runs, the decay goes into the shared base for every model. Changing the model size or the data (layers, pairs, memory tokens, examples) is refused unless `ALLOW_SIZE_CHANGE = True`, so all runs stay comparable.

`NOTE` is one sentence on why you launch this run. It goes into `runs.md` and the `why` column of `results.csv`, and it can change between sessions of the same run.

In [ ]:
RUN = 'P_rmt_L4_p16_1M_s1'
NOTE = 'Pilot. Does RMT with the reconstruction loss learn 16 pairs?'
RUN_TYPE = 'rmt'            # 'rmt', 'gradmem' (from an RMT parent) or 'gradmem_reference' (from scratch)
K = 1                       # rmt: passes over the context. gradmem: write steps
PARENT_RMT_RUN = None       # gradmem: required, e.g. 'E1_rmt_L4_p16_1M_s1'. rmt: optional warm start
PARENT_STEP = None          # None takes the parent's newest checkpoint on Drive
EXTRA = dict(max_steps=30000,   # arguments for this run only
             lr_scheduler_type='linear')   # pilot test of learning-rate decay (to 1/10 of the rate at the end)
ALLOW_SIZE_CHANGE = False   # True only on purpose, it breaks comparability with the other runs

In [ ]:
import json, os, re, glob

# ---------- the shared base (plan, D11 to D13) ----------
BASE = dict(
    # data (MQAR), 16 pairs, 1M examples
    vocab_size=8192, num_kv_pairs=16, input_seq_len=48,
    query_sampling='uniform', power_a=0.01,
    train_num_examples=1000000, valid_num_examples=3000, data_seed=123,
    # model, 4 layers, hidden 128
    base_model='llama', n_layer=4, n_head=4, n_embd=128, attn_implementation='eager',
    n_mem_tokens=8,
    # training
    per_device_batch_size=64, gradient_accumulation_steps=1, total_batch_size=64,
    learning_rate=1e-4, max_steps=60000, warmup_steps=2000,
    eval_steps=500, logging_steps=500,
    early_stopping_patience=500, stop_on_metric_value=0.99,
    seed=143,
)
RMT_PART = dict(K=K, use_reconstruction_loss=True, reconstruction_loss_weight=1.0)
GRADMEM_PART = dict(memory_backend='prefix', K=K, last_K_second_order=K, inner_lr=0.4,
                    use_adam=False, grad_mode='second', freeze_backbone=False)
SIZE_KEYS = ['vocab_size', 'num_kv_pairs', 'input_seq_len', 'n_layer', 'n_head', 'n_embd',
             'n_mem_tokens', 'train_num_examples', 'data_seed']
MODEL_KEYS = ['vocab_size', 'n_layer', 'n_head', 'n_embd', 'n_mem_tokens', 'num_kv_pairs']

def parent_checkpoint(parent, step, cfg):
    # check the RMT parent and return the path of the checkpoint to start from
    pdir = f'{PROJ}/runs/{parent}'
    if not os.path.exists(f'{pdir}/config.json'):
        raise ValueError(f'Parent {parent} not found on Drive ({pdir}).')
    rec = json.load(open(f'{pdir}/config.json'))
    if rec.get('script') != 'run_rmt_on_mqar.py':
        raise ValueError(f'Parent {parent} is not an RMT run (script {rec.get("script")}).')
    pcfg = rec['config']
    if not pcfg.get('use_reconstruction_loss'):
        raise ValueError(f'Parent {parent} was trained without the reconstruction loss (plan, D11).')
    diff = {k: (pcfg.get(k), cfg.get(k)) for k in MODEL_KEYS if pcfg.get(k) != cfg.get(k)}
    if diff and not ALLOW_SIZE_CHANGE:
        raise ValueError(f'Parent {parent} has a different model size or pair count: {diff}')
    cks = {int(m.group(1)): p for p in glob.glob(f'{pdir}/checkpoints/checkpoint-*/model.safetensors')
           for m in [re.search(r'checkpoint-(\d+)', p)] if m}
    if not cks:
        raise ValueError(f'Parent {parent} has no checkpoint on Drive yet.')
    chosen = step if step is not None else max(cks)
    if chosen not in cks:
        raise ValueError(f'Parent {parent} has no checkpoint-{chosen}. Available {sorted(cks)}')
    summary = f'{pdir}/summary.json'
    status = json.load(open(summary)).get('status') if os.path.exists(summary) else 'unknown'
    if status not in ('finished', 'stopped'):
        print(f'Note. Parent {parent} has status "{status}". Its best checkpoint may still change.')
    print(f'Parent {parent}, checkpoint-{chosen}')
    return cks[chosen]

# ---------- build the configuration ----------
if RUN_TYPE == 'rmt':
    SCRIPT = 'run_rmt_on_mqar.py'
    CONFIG = {**BASE, **RMT_PART}
elif RUN_TYPE in ('gradmem', 'gradmem_reference'):
    SCRIPT = 'run_gradmemgpt_on_mqar.py'
    CONFIG = {**BASE, **GRADMEM_PART}
else:
    raise ValueError("RUN_TYPE must be 'rmt', 'gradmem' or 'gradmem_reference'")

bad = [k for k in EXTRA if k in SIZE_KEYS and EXTRA[k] != CONFIG.get(k)]
if bad and not ALLOW_SIZE_CHANGE:
    raise ValueError(f'EXTRA changes the model size or data {bad}. Set ALLOW_SIZE_CHANGE = True only on purpose.')
if 'init_base_checkpoint' in EXTRA:
    raise ValueError('init_base_checkpoint loads nothing for RMT checkpoints. The parent is set by PARENT_RMT_RUN.')
if RUN_TYPE == 'rmt' and EXTRA.get('use_reconstruction_loss') is False:
    raise ValueError('RMT without the reconstruction loss stays at the copy baseline (plan, D11).')
CONFIG.update(EXTRA)

LOCAL_DIR = f'{LOCAL_RUNS}/{RUN}'
DRIVE_DIR = f'{PROJ}/runs/{RUN}'
saved = f'{DRIVE_DIR}/config.json'
old = json.load(open(saved)) if os.path.exists(saved) else None

if RUN_TYPE == 'gradmem' and not PARENT_RMT_RUN:
    raise ValueError("A 'gradmem' run must start from RMT weights. Set PARENT_RMT_RUN (plan, D1, D10).")
if RUN_TYPE == 'gradmem_reference' and PARENT_RMT_RUN:
    raise ValueError("'gradmem_reference' runs train from scratch. Use RUN_TYPE 'gradmem' to start from RMT.")
if PARENT_RMT_RUN:
    if old is not None and 'init_checkpoint' in old['config']:
        CONFIG['init_checkpoint'] = old['config']['init_checkpoint']   # a resumed run keeps its original start
        print('Resumed run, keeping its original start', CONFIG['init_checkpoint'])
    else:
        CONFIG['init_checkpoint'] = parent_checkpoint(PARENT_RMT_RUN, PARENT_STEP, CONFIG)
    if 'warmup_steps' not in EXTRA:
        CONFIG['warmup_steps'] = 1000
elif RUN_TYPE == 'gradmem_reference':
    print('Reference run from scratch (E2 or E3).')

# ---------- one name, one configuration ----------
def to_args(cfg):
    out = []
    for k, v in cfg.items():
        if v is None:
            continue
        out += [f'--{k}', ('true' if v else 'false') if isinstance(v, bool) else str(v)]
    return out

os.makedirs(DRIVE_DIR, exist_ok=True)
record = {'script': SCRIPT, 'config': CONFIG}
if old is not None:
    if old != json.loads(json.dumps(record)):
        raise ValueError(f'{RUN} already exists on Drive with a different configuration. '
                         'Use a new RUN name, or restore the old configuration.')
    print('Same configuration as before, the run will resume.')
else:
    json.dump(record, open(saved, 'w'), indent=2)
    print('New run, configuration saved to', saved)

with open(f'{DRIVE_DIR}/code_versions.txt', 'a') as f:
    f.write(f'{BRANCH} {COMMIT}\n')

ARGS = ['--exp_path', LOCAL_DIR] + to_args(CONFIG)
print(SCRIPT, ' '.join(ARGS))

## 7. Start or resume the run

The run starts in the background, so the notebook stays usable. The first lines of the log say whether it resumed and from which step. To run two configurations at once on one T4, run cells 6 to 7 again with another `RUN` name.

Each session writes its own console log file, so logs from different sessions never overwrite each other.

The cell refuses to start a run whose name is still running in this session, because two copies would train into the same folder and overwrite each other.

In [ ]:
import subprocess, time, os

def run_is_alive(name):
    # True if a run with this name was started in this session and is still running
    pid_file = f'/content/logs/{name}.pid'
    if not os.path.exists(pid_file):
        return False
    pid = int(open(pid_file).read())
    try:
        state = open(f'/proc/{pid}/stat').read().rsplit(')', 1)[1].split()[0]
    except FileNotFoundError:
        return False
    return state != 'Z'                   # 'Z' means finished but not yet collected

if run_is_alive(RUN):
    raise RuntimeError(f'{RUN} is still running. Two copies would overwrite each other. '
                       'Watch it in cell 8, or stop it in cell 9 first.')

os.makedirs('/content/logs', exist_ok=True)
LOG =f'/content/logs/{RUN}_{time.strftime("%Y%m%d-%H%M%S")}.log'
cmd = ['python', '-u', '/content/colab_run.py',
       '--repo', REPO, '--script', SCRIPT,
       '--local_dir', LOCAL_DIR, '--drive_dir', DRIVE_DIR,
       '--log', LOG, '--sync_minutes', str(SYNC_MINUTES),
       '--note', NOTE, '--commit', f'{BRANCH} {COMMIT}', '--run_type', RUN_TYPE,
       '--runs_md', f'{PROJ}/results/runs.md',
       '--results_csv', f'{PROJ}/results/results.csv', '--'] + ARGS

proc = subprocess.Popen(cmd, stdout=open(LOG, 'w'), stderr=subprocess.STDOUT, start_new_session=True)
open(f'/content/logs/{RUN}.pid', 'w').write(str(proc.pid))
open(f'/content/logs/{RUN}.current', 'w').write(LOG)
print('Started', RUN, 'pid', proc.pid, 'log', LOG)

## 8. Watch progress

Shows the last lines of the console log of the chosen run. The TensorBoard cell shows the curves of all local runs.

In [ ]:
WATCH = RUN
log = open(f'/content/logs/{WATCH}.current').read().strip()
!tail -n 15 "{log}"

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/runs

## 9. Stop a run

Stops one run. Type its name into `STOP` and run the cell. The helper then mirrors the run to Drive one last time, so nothing saved is lost. With `STOP` empty the cell does nothing, so "Run all" never stops a run by accident.

In [ ]:
import os, signal
STOP = ''   # name of the run to stop, for example 'E3_gm_k2_p16_s1'

if not STOP:
    print('Nothing stopped. Type a run name into STOP to stop it.')
else:
    pid = int(open(f'/content/logs/{STOP}.pid').read())
    try:
        os.killpg(pid, signal.SIGTERM)
        print('Sent stop signal to', STOP)
    except ProcessLookupError:
        print(STOP, 'is not running')

## 10. Results of all runs on Drive

Rebuilds `results/results.csv` from every run folder on Drive and shows it. Running runs also rebuild it at every mirror, so this cell is only needed to see the table now.

| Column | Meaning |
|---|---|
| `status` | running, finished, stopped or failed. A `running` run with an old `last_update` was cut by a disconnect |
| `init_from` | scratch, or the checkpoint a curriculum run started from |
| `best_acc`, `step_to_99` | the best eval accuracy and the first eval step that reached 99% |
| `ms_per_step` | wall time per training step over all sessions, eval time included |
| `why`, `conclusion` | your newest launch note and your conclusion |

In [ ]:
import sys
import pandas as pd
sys.path.insert(0, '/content')
import importlib, colab_run
importlib.reload(colab_run)

colab_run.rebuild_results(f'{PROJ}/runs', f'{PROJ}/results/results.csv')
pd.set_option('display.max_colwidth', 80)
pd.read_csv(f'{PROJ}/results/results.csv')

## 10b. Add a conclusion

After a run ends, write what you learned from it in one or two sentences. It goes into `runs.md` as a new entry and into the `conclusion` column of `results.csv`. Running this cell again for the same run replaces the conclusion in the table, and `runs.md` keeps every version in order.

In [ ]:
import json, sys, time
from pathlib import Path
sys.path.insert(0, '/content')
import importlib, colab_run
importlib.reload(colab_run)

CONCLUDE_RUN = RUN
CONCLUSION = ''   # for example 'Reached 99% at step 21k, so the Zoology-size setup works and phase 2 can start.'

if not CONCLUSION.strip():
    print('Nothing saved. Write a conclusion into CONCLUSION to save it.')
else:
    summary_path = Path(f'{PROJ}/runs/{CONCLUDE_RUN}/summary.json')
    summary = json.load(open(summary_path)) if summary_path.exists() else {'run': CONCLUDE_RUN, 'sessions': []}
    summary['conclusion'] = CONCLUSION
    colab_run.write_json_atomic(summary_path, summary)
    colab_run.append_md(f'{PROJ}/results/runs.md',
                        f'### {time.strftime("%Y-%m-%d %H:%M")} · {CONCLUDE_RUN} · conclusion\n- {CONCLUSION}')
    colab_run.rebuild_results(f'{PROJ}/runs', f'{PROJ}/results/results.csv')
    print('Conclusion saved for', CONCLUDE_RUN)

## 11. Before you close the session

Stops every run that is still training. Each run then makes its own final mirror to Drive, and this cell waits for that. After that all writes are flushed to Drive. Runs lose at most the steps since their last checkpoint (500 steps).

Set `CLOSE = True` to use it. With `False` the cell does nothing, so "Run all" never stops your runs.

In [ ]:
import glob, os, signal, time
CLOSE = False   # set to True only when you are about to close the session

def alive(pid):
    try:
        state = open(f'/proc/{pid}/stat').read().rsplit(')', 1)[1].split()[0]
    except FileNotFoundError:
        return False
    if state == 'Z':                      # finished, but not yet collected by the notebook
        try:
            os.waitpid(pid, os.WNOHANG)
        except ChildProcessError:
            pass
        return False
    return True

def close_session():
    running = []
    for pid_file in glob.glob('/content/logs/*.pid'):
        pid = int(open(pid_file).read())
        if alive(pid):
            os.killpg(pid, signal.SIGTERM)
            running.append((os.path.basename(pid_file)[:-4], pid))

    deadline = time.time() + 600
    while running and time.time() < deadline:
        running = [(n, p) for n, p in running if alive(p)]
        time.sleep(5)
    print('Still running after 10 minutes:', running if running else 'none')

    from google.colab import drive
    drive.flush_and_unmount()
    print('Drive writes finished. You can close the session.')

if CLOSE:
    close_session()
else:
    print('Nothing closed. Set CLOSE = True to stop all runs and finish Drive writes.')